# Phase 1 — Data Setup & Exploration

**Project:** Community Notes / Misinformation  
**Owner:** Jiamao Tian  
**Task:** Task 1 — Data Setup & Exploration

## Purpose

This notebook documents the Phase 1 data exploration for the four core Community Notes tables used in our project:

1. **Notes**
2. **Note Request**
3. **Ratings**
4. **Note Status History**

To keep the exploration manageable and consistent, the notebook reads the **first 1,000 rows from each table** using pandas.

> These are real rows from the downloaded public datasets. The raw files are kept locally in `project/data/` and are **not committed to GitHub**.

**Important:** The first 1,000 rows are used for schema and feasibility checks. They are not a random sample and should not be treated as representative of the full Community Notes population.

**Source:** X Community Notes Public Data  
https://communitynotes.x.com/guide/en/under-the-hood/download-data


## 1. Setup

Expected local structure:

```text
project/
├── phase1_data_exploration.ipynb
└── data/
    ├── notes-....tsv
    ├── batSignals-00000.tsv
    ├── ratings-00007.tsv
    └── noteStatusHistory-00000.tsv
```

The `data/` folder should remain excluded from Git using `.gitignore`.


In [55]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data")
SAMPLE_N = 1_000

def first_matching_file(pattern):
    matches = sorted(DATA_DIR.glob(pattern))
    if not matches:
        raise FileNotFoundError(
            f"No file matching {pattern!r} was found in {DATA_DIR.resolve()}"
        )
    return matches[0]

NOTES_FILE = first_matching_file("notes-*.tsv")
REQUESTS_FILE = DATA_DIR / "batSignals-00000.tsv"
RATINGS_FILE = DATA_DIR / "ratings-00007.tsv"
STATUS_FILE = DATA_DIR / "noteStatusHistory-00000.tsv"

print("Notes file:", NOTES_FILE)
print("Note Request file:", REQUESTS_FILE)
print("Ratings file:", RATINGS_FILE)
print("Note Status History file:", STATUS_FILE)


Notes file: data/notes-00001.tsv
Note Request file: data/batSignals-00000.tsv
Ratings file: data/ratings-00007.tsv
Note Status History file: data/noteStatusHistory-00000.tsv


## 2. Notes — 1,000-row manageable sample

The Notes table provides the bridge between post-level request data and note-level rating/status data because it contains both `tweetId` and `noteId`.

For Phase 1, one row is interpreted as one Community Note record in the inspected Notes data.


In [56]:
notes_sample = pd.read_csv(
    NOTES_FILE,
    sep="\t",
    nrows=SAMPLE_N
)

print("Sample shape:", notes_sample.shape)
print("Number of columns:", len(notes_sample.columns))
print("\nColumns:")
print(notes_sample.columns.tolist())

notes_sample.head()


Sample shape: (1000, 24)
Number of columns: 24

Columns:
['noteId', 'noteAuthorParticipantId', 'createdAtMillis', 'tweetId', 'classification', 'believable', 'harmful', 'validationDifficulty', 'misleadingOther', 'misleadingFactualError', 'misleadingManipulatedMedia', 'misleadingOutdatedInformation', 'misleadingMissingImportantContext', 'misleadingUnverifiedClaimAsFact', 'misleadingSatire', 'notMisleadingOther', 'notMisleadingFactuallyCorrect', 'notMisleadingOutdatedButNotWhenWritten', 'notMisleadingClearlySatire', 'notMisleadingPersonalOpinion', 'trustworthySources', 'summary', 'isMediaNote', 'isCollaborativeNote']


,noteId,noteAuthorParticipantId,createdAtMillis,tweetId,classification,believable,harmful,validationDifficulty,misleadingOther,misleadingFactualError,...,misleadingSatire,notMisleadingOther,notMisleadingFactuallyCorrect,notMisleadingOutdatedButNotWhenWritten,notMisleadingClearlySatire,notMisleadingPersonalOpinion,trustworthySources,summary,isMediaNote,isCollaborativeNote
0,2076363424486076857,F75B04680623CD9A452496D4AB69A20ED730CE93652C71...,1783878697379,17453093548,NOT_MISLEADING,NaN,NaN,NaN,0,0,...,0,0,0,0,0,0,0,This post dates to July 2010. It promoted the ...,0,1
1,2096040661472469421,F75B04680623CD9A452496D4AB69A20ED730CE93652C71...,1788570144074,255751329392234496,MISINFORMED_OR_POTENTIALLY_MISLEADING,NaN,NaN,NaN,0,0,...,0,0,0,0,0,0,0,This was the attendance for Manchester City's ...,0,1
2,2067936356123480310,F75B04680623CD9A452496D4AB69A20ED730CE93652C71...,1781869705697,296050305089552384,MISINFORMED_OR_POTENTIALLY_MISLEADING,NaN,NaN,NaN,0,0,...,0,0,0,0,0,0,0,"The image shows Amala and Ewedu soup, a tradit...",0,1
3,2091873394211844466,F75B04680623CD9A452496D4AB69A20ED730CE93652C71...,1787576591021,492989518664847360,MISINFORMED_OR_POTENTIALLY_MISLEADING,NaN,NaN,NaN,0,0,...,0,0,0,0,0,0,0,Hamas members carried out the kidnapping and m...,0,1
4,2092780222446006614,F75B04680623CD9A452496D4AB69A20ED730CE93652C71...,1787793074788,1093126770025480192,MISINFORMED_OR_POTENTIALLY_MISLEADING,NaN,NaN,NaN,0,0,...,0,0,0,0,0,0,0,The photo appears to be a digitally altered ve...,0,1


In [57]:
notes_key_fields = [c for c in [
    "noteId",
    "tweetId",
    "noteAuthorParticipantId",
    "createdAtMillis",
    "classification"
] if c in notes_sample.columns]

notes_sample[notes_key_fields].head(10)


,noteId,tweetId,noteAuthorParticipantId,createdAtMillis,classification
0,2076363424486076857,17453093548,F75B04680623CD9A452496D4AB69A20ED730CE93652C71...,1783878697379,NOT_MISLEADING
1,2096040661472469421,255751329392234496,F75B04680623CD9A452496D4AB69A20ED730CE93652C71...,1788570144074,MISINFORMED_OR_POTENTIALLY_MISLEADING
2,2067936356123480310,296050305089552384,F75B04680623CD9A452496D4AB69A20ED730CE93652C71...,1781869705697,MISINFORMED_OR_POTENTIALLY_MISLEADING
3,2091873394211844466,492989518664847360,F75B04680623CD9A452496D4AB69A20ED730CE93652C71...,1787576591021,MISINFORMED_OR_POTENTIALLY_MISLEADING
4,2092780222446006614,1093126770025480192,F75B04680623CD9A452496D4AB69A20ED730CE93652C71...,1787793074788,MISINFORMED_OR_POTENTIALLY_MISLEADING
5,2082124719525130520,1253353708961902596,F75B04680623CD9A452496D4AB69A20ED730CE93652C71...,1785252350875,MISINFORMED_OR_POTENTIALLY_MISLEADING
6,2062018081854202070,1286632257076289536,F75B04680623CD9A452496D4AB69A20ED730CE93652C71...,1780458426732,NOT_MISLEADING
7,2080677948072632451,1325481577586057218,F75B04680623CD9A452496D4AB69A20ED730CE93652C71...,1784907395909,NOT_MISLEADING
8,2083018000613970107,1344588336288190464,F75B04680623CD9A452496D4AB69A20ED730CE93652C71...,1785465404094,MISINFORMED_OR_POTENTIALLY_MISLEADING
9,2061495122693701890,1401375813035978752,F75B04680623CD9A452496D4AB69A20ED730CE93652C71...,1780333831167,NOT_MISLEADING


## 3. Note Request — 1,000-row manageable sample

The Note Request data contains post-level request eligibility information.

A row represents a post/tweet associated with Community Notes request eligibility data. The main join key for our project is `tweetId`.


In [58]:
requests_sample = pd.read_csv(
    REQUESTS_FILE,
    sep="\t",
    nrows=SAMPLE_N
)

print("Sample shape:", requests_sample.shape)
print("Number of columns:", len(requests_sample.columns))
print("\nColumns:")
print(requests_sample.columns.tolist())

requests_sample.head()


Sample shape: (1000, 7)
Number of columns: 7

Columns:
['tweetId', 'noteRequestFeedEligibleAtMillis', 'apiSmallFeedEligibleAtMillis', 'apiLargeFeedEligibleAtMillis', 'apiXlFeedEligibleAtMillis', 'sourceLinks', 'suggestions']


,tweetId,noteRequestFeedEligibleAtMillis,apiSmallFeedEligibleAtMillis,apiLargeFeedEligibleAtMillis,apiXlFeedEligibleAtMillis,sourceLinks,suggestions
0,442487652,-1,-1,-1,1783993244141,NaN,[]
1,1316941803,-1,-1,1785174077386,1773211963793,NaN,[]
2,1463732880,-1,-1,-1,1787257421863,NaN,[]
3,1711529307,-1,-1,-1,1788047004994,NaN,[]
4,4176682202,-1,-1,-1,1786803953724,NaN,"[{""suggestion_id"":2088632163939832178,""suggest..."


In [59]:
request_key_fields = [c for c in [
    "tweetId",
    "noteRequestFeedEligibleAtMillis",
    "apiSmallFeedEligibleAtMillis",
    "apiLargeFeedEligibleAtMillis",
    "apiXlFeedEligibleAtMillis",
    "sourceLinks",
    "suggestions"
] if c in requests_sample.columns]

requests_sample[request_key_fields].head(10)


,tweetId,noteRequestFeedEligibleAtMillis,apiSmallFeedEligibleAtMillis,apiLargeFeedEligibleAtMillis,apiXlFeedEligibleAtMillis,sourceLinks,suggestions
0,442487652,-1,-1,-1,1783993244141,NaN,[]
1,1316941803,-1,-1,1785174077386,1773211963793,NaN,[]
2,1463732880,-1,-1,-1,1787257421863,NaN,[]
3,1711529307,-1,-1,-1,1788047004994,NaN,[]
4,4176682202,-1,-1,-1,1786803953724,NaN,"[{""suggestion_id"":2088632163939832178,""suggest..."
5,17453093548,-1,-1,1783878799895,1783719653481,NaN,[]
6,6041684420853760,-1,-1,-1,1784085426592,NaN,[]
7,14510999067435008,-1,-1,-1,1786353072302,NaN,[]
8,57314959016726529,-1,-1,-1,1788478029232,NaN,[]
9,70215858424782848,-1,-1,-1,1786700843075,NaN,[]


## 4. Ratings — 1,000-row manageable sample

The Ratings table was the main additional table inspected after the instructor feedback.

A row represents one rating submitted by one participant for one Community Note. The main note-level join key is `noteId`.


In [60]:
ratings_sample = pd.read_csv(
    RATINGS_FILE,
    sep="\t",
    nrows=SAMPLE_N
)

print("Sample shape:", ratings_sample.shape)
print("Number of columns:", len(ratings_sample.columns))
print("\nColumns:")
print(ratings_sample.columns.tolist())

ratings_sample.head()


Sample shape: (1000, 35)
Number of columns: 35

Columns:
['noteId', 'raterParticipantId', 'createdAtMillis', 'version', 'agree', 'disagree', 'helpful', 'notHelpful', 'helpfulnessLevel', 'helpfulOther', 'helpfulInformative', 'helpfulClear', 'helpfulEmpathetic', 'helpfulGoodSources', 'helpfulUniqueContext', 'helpfulAddressesClaim', 'helpfulImportantContext', 'helpfulUnbiasedLanguage', 'notHelpfulOther', 'notHelpfulIncorrect', 'notHelpfulSourcesMissingOrUnreliable', 'notHelpfulOpinionSpeculationOrBias', 'notHelpfulMissingKeyPoints', 'notHelpfulOutdated', 'notHelpfulHardToUnderstand', 'notHelpfulArgumentativeOrBiased', 'notHelpfulOffTopic', 'notHelpfulSpamHarassmentOrAbuse', 'notHelpfulIrrelevantSources', 'notHelpfulOpinionSpeculation', 'notHelpfulNoteNotNeeded', 'ratedOnTweetId', 'ratingSourceBucketed', 'suggestion', 'suggestionId']


,noteId,raterParticipantId,createdAtMillis,version,agree,disagree,helpful,notHelpful,helpfulnessLevel,helpfulOther,...,notHelpfulArgumentativeOrBiased,notHelpfulOffTopic,notHelpfulSpamHarassmentOrAbuse,notHelpfulIrrelevantSources,notHelpfulOpinionSpeculation,notHelpfulNoteNotNeeded,ratedOnTweetId,ratingSourceBucketed,suggestion,suggestionId
0,2021294138574962942,C4F546539DE2DAD1DE53FAAE782A1A2AE4D584E486738F...,1770753267906,2,0,0,0,0,HELPFUL,0,...,0,0,0,0,0,0,2021236742460756143,DEFAULT,NaN,-1
1,2021294138574962942,1F389E4D82F9365D68F08DE88B2183B62495C0A02A5647...,1770749606640,2,0,0,0,0,HELPFUL,0,...,0,0,0,0,0,0,2021236742460756143,DEFAULT,NaN,-1
2,2021294138574962942,9DBF0CB45CE2DF281018C109FA5A2EF5B567978B31FD7B...,1770751069049,2,0,0,0,0,HELPFUL,0,...,0,0,0,0,0,0,2021236742460756143,DEFAULT,NaN,-1
3,2021294138574962942,CACA67F7ABC2A5F1387D0FC3C036C86D7A49037AF1CE58...,1770757293660,2,0,0,0,0,HELPFUL,0,...,0,0,0,0,0,0,2021236742460756143,DEFAULT,NaN,-1
4,2021294138574962942,13DCD9F28E1D9129D3CB9BAA2ED08FF90AADE37E4A0B03...,1770770877458,2,0,0,0,0,HELPFUL,1,...,0,0,0,0,0,0,2021236742460756143,DEFAULT,NaN,-1


In [61]:
ratings_key_fields = [c for c in [
    "noteId",
    "raterParticipantId",
    "createdAtMillis",
    "helpfulnessLevel"
] if c in ratings_sample.columns]

ratings_sample[ratings_key_fields].head(10)


,noteId,raterParticipantId,createdAtMillis,helpfulnessLevel
0,2021294138574962942,C4F546539DE2DAD1DE53FAAE782A1A2AE4D584E486738F...,1770753267906,HELPFUL
1,2021294138574962942,1F389E4D82F9365D68F08DE88B2183B62495C0A02A5647...,1770749606640,HELPFUL
2,2021294138574962942,9DBF0CB45CE2DF281018C109FA5A2EF5B567978B31FD7B...,1770751069049,HELPFUL
3,2021294138574962942,CACA67F7ABC2A5F1387D0FC3C036C86D7A49037AF1CE58...,1770757293660,HELPFUL
4,2021294138574962942,13DCD9F28E1D9129D3CB9BAA2ED08FF90AADE37E4A0B03...,1770770877458,HELPFUL
5,2021294138574962942,DB2B2C561B44A2D6D4C0ED4E0A019F4E8891C62262EC4E...,1770762078991,HELPFUL
6,2021294138574962942,9A0F58898520C246E0511D1721A60364D02CC2809E2656...,1770769473506,HELPFUL
7,2021294138574962942,C5CBEAAB89CB5FE91D2DA140B2DFF5E8DD2CEB2673EA4D...,1770751228286,HELPFUL
8,2021294138574962942,D8B2DA741A9AB23819166EBF1D4D895B6DC694714784A3...,1770751657717,HELPFUL
9,2021294138574962942,16C82BE50B00D591BDB8AF3F5C4B57A1C7A3CF631479E8...,1770756625541,HELPFUL


In [62]:
print("Unique notes:", ratings_sample["noteId"].nunique())
print("Unique raters:", ratings_sample["raterParticipantId"].nunique())

if "helpfulnessLevel" in ratings_sample.columns:
    print("\nHelpfulness levels:")
    print(ratings_sample["helpfulnessLevel"].value_counts(dropna=False))

if "createdAtMillis" in ratings_sample.columns:
    rating_dates = pd.to_datetime(
        ratings_sample["createdAtMillis"],
        unit="ms",
        errors="coerce"
    )
    print("\nRating date range in this 1,000-row sample:")
    print(rating_dates.min(), "to", rating_dates.max())


Unique notes: 10
Unique raters: 998

Helpfulness levels:
helpfulnessLevel
HELPFUL             821
NOT_HELPFUL         165
SOMEWHAT_HELPFUL     14
Name: count, dtype: int64

Rating date range in this 1,000-row sample:
2026-02-10 18:44:33.207000 to 2026-04-05 18:03:42.313000


## 5. Note Status History — 1,000-row manageable sample

The Note Status History table contains note-level status information and status-related timestamps.

For Phase 1, one row is treated as a status-history record for a Community Note identified by `noteId`. The exact status fields available in the downloaded file are inspected below rather than assumed in advance.


In [63]:
status_sample = pd.read_csv(
    STATUS_FILE,
    sep="\t",
    nrows=SAMPLE_N
)

print("Sample shape:", status_sample.shape)
print("Number of columns:", len(status_sample.columns))
print("\nColumns:")
print(status_sample.columns.tolist())

status_sample.head()


Sample shape: (1000, 23)
Number of columns: 23

Columns:
['noteId', 'noteAuthorParticipantId', 'createdAtMillis', 'timestampMillisOfFirstNonNMRStatus', 'firstNonNMRStatus', 'timestampMillisOfCurrentStatus', 'currentStatus', 'timestampMillisOfLatestNonNMRStatus', 'mostRecentNonNMRStatus', 'timestampMillisOfStatusLock', 'lockedStatus', 'timestampMillisOfRetroLock', 'currentCoreStatus', 'currentExpansionStatus', 'currentGroupStatus', 'currentDecidedBy', 'currentModelingGroup', 'timestampMillisOfMostRecentStatusChange', 'timestampMillisOfNmrDueToMinStableCrhTime', 'currentMultiGroupStatus', 'currentModelingMultiGroup', 'timestampMinuteOfFinalScoringOutput', 'timestampMillisOfFirstNmrDueToMinStableCrhTime']


,noteId,noteAuthorParticipantId,createdAtMillis,timestampMillisOfFirstNonNMRStatus,firstNonNMRStatus,timestampMillisOfCurrentStatus,currentStatus,timestampMillisOfLatestNonNMRStatus,mostRecentNonNMRStatus,timestampMillisOfStatusLock,...,currentExpansionStatus,currentGroupStatus,currentDecidedBy,currentModelingGroup,timestampMillisOfMostRecentStatusChange,timestampMillisOfNmrDueToMinStableCrhTime,currentMultiGroupStatus,currentModelingMultiGroup,timestampMinuteOfFinalScoringOutput,timestampMillisOfFirstNmrDueToMinStableCrhTime
0,1352796878438424576,B021517F4F8F9F372DCD70BB8F8A21A80C12D7EFDE8B2F...,1611366884227,NaN,NaN,1721920068424,NEEDS_MORE_RATINGS,NaN,NaN,1674003109873,...,NEEDS_MORE_RATINGS,NEEDS_MORE_RATINGS,CoreModel (v1.1),13.0,-1,NaN,NaN,NaN,29818310,NaN
1,1353415873227177985,D4C9EDB464DF2D4A6D8F68CA8D1F86226B50EB9D3B0F33...,1611514464087,NaN,NaN,1721920068424,NEEDS_MORE_RATINGS,NaN,NaN,1674003109873,...,NEEDS_MORE_RATINGS,NEEDS_MORE_RATINGS,CoreModel (v1.1),13.0,-1,NaN,NaN,NaN,29818310,NaN
2,1354586938863443971,C0AF45F4C4B2240E7AB31456957E5D770FF7AFA13627C7...,1611793667892,NaN,NaN,1781032339438,NEEDS_MORE_RATINGS,NaN,NaN,1674003109873,...,NEEDS_MORE_RATINGS,NaN,CoreModel (v1.1),NaN,-1,-1.0,NaN,NaN,29818310,NaN
3,1354588003075764229,9C480F7B2E882F19F9FBF75BD13FEA53031875AC42FDF6...,1611793921619,NaN,NaN,1721920068424,NEEDS_MORE_RATINGS,NaN,NaN,1674003109873,...,NaN,NaN,ExpansionPlusModel (v1.1),NaN,-1,NaN,NaN,NaN,29818310,NaN
4,1354588172659920899,931B6CFE7C2FDF3653B0A63DC20DF3C51F8AA3BEB9D613...,1611793962051,NaN,NaN,1721920068424,NEEDS_MORE_RATINGS,NaN,NaN,1674003109873,...,NaN,NaN,ExpansionPlusModel (v1.1),NaN,-1,NaN,NaN,NaN,29818310,NaN


In [64]:
status_key_fields = [c for c in [
    "noteId",
    "noteAuthorParticipantId",
    "createdAtMillis",
    "currentStatus",
    "firstNonNMRStatus",
    "mostRecentNonNMRStatus"
] if c in status_sample.columns]

status_sample[status_key_fields].head(10)


,noteId,noteAuthorParticipantId,createdAtMillis,currentStatus,firstNonNMRStatus,mostRecentNonNMRStatus
0,1352796878438424576,B021517F4F8F9F372DCD70BB8F8A21A80C12D7EFDE8B2F...,1611366884227,NEEDS_MORE_RATINGS,NaN,NaN
1,1353415873227177985,D4C9EDB464DF2D4A6D8F68CA8D1F86226B50EB9D3B0F33...,1611514464087,NEEDS_MORE_RATINGS,NaN,NaN
2,1354586938863443971,C0AF45F4C4B2240E7AB31456957E5D770FF7AFA13627C7...,1611793667892,NEEDS_MORE_RATINGS,NaN,NaN
3,1354588003075764229,9C480F7B2E882F19F9FBF75BD13FEA53031875AC42FDF6...,1611793921619,NEEDS_MORE_RATINGS,NaN,NaN
4,1354588172659920899,931B6CFE7C2FDF3653B0A63DC20DF3C51F8AA3BEB9D613...,1611793962051,NEEDS_MORE_RATINGS,NaN,NaN
5,1354589054889820168,F35972BBD2F99515FD974E9C7AFD899970F2E4A5911513...,1611794172392,NEEDS_MORE_RATINGS,NaN,NaN
6,1354589707716481025,76C37E2A80FF9B994DDC4762345A2C54A4B47BBD07479C...,1611794328036,NEEDS_MORE_RATINGS,NaN,NaN
7,1354590276954816513,A9AA713BD9E7C6FEED40C8B7432F7DCAC809F77018E33C...,1611794463755,NEEDS_MORE_RATINGS,NaN,NaN
8,1354590349381881857,4DB1A77C8620B694EDA9352D75DC667DCD17BDDD17DE7B...,1611794481021,NEEDS_MORE_RATINGS,NaN,NaN
9,1354590891764293637,3DC9E84FD4964D9A3FDF80101F73238C6E12DC680AAFB3...,1611794610337,NEEDS_MORE_RATINGS,NaN,NaN


## 6. How the four tables connect

The proposed project funnel is:

**Request → Note Written → Ratings Received → Status**

The main join structure is:

```text
Note Request
    tweetId
       ↓
Notes
    tweetId + noteId
       ↓
Ratings
    noteId

Notes
    noteId
       ↓
Note Status History
    noteId
```

This structure allows the team to investigate where requests or notes appear to get stuck in the funnel.


In [65]:
join_check = pd.DataFrame({
    "Table": [
        "Notes",
        "Note Request",
        "Ratings",
        "Note Status History"
    ],
    "Sample rows": [
        len(notes_sample),
        len(requests_sample),
        len(ratings_sample),
        len(status_sample)
    ],
    "Columns": [
        len(notes_sample.columns),
        len(requests_sample.columns),
        len(ratings_sample.columns),
        len(status_sample.columns)
    ],
    "Primary project key(s)": [
        "tweetId, noteId",
        "tweetId",
        "noteId",
        "noteId"
    ]
})

join_check


,Table,Sample rows,Columns,Primary project key(s)
0,Notes,1000,24,"tweetId, noteId"
1,Note Request,1000,7,tweetId
2,Ratings,1000,35,noteId
3,Note Status History,1000,23,noteId


## 7. Initial data-fit findings

The four inspected tables provide a feasible starting point for the project funnel:

- **Note Request + Notes** can support analysis of whether a request has an observed note and can help define a candidate time-to-first-note metric.
- **Ratings** can support analysis of rating volume, rating timing, and helpfulness evaluations.
- **Note Status History** can support analysis of note status and status-related timing.
- **Notes** connects the post-level and note-level parts of the data through `tweetId` and `noteId`.

These tables can therefore support candidate dashboard measures such as:

- unanswered requests
- time to first note
- rating volume
- unresolved notes
- time to status

### Important limitations

- These 1,000-row samples are used for **schema and feasibility checks**, not population estimates.
- Because the samples use the first 1,000 rows, they are **not random samples**.
- Rating count alone does not establish broad agreement.
- Final definitions of **coverage gap**, **response time**, **enough ratings**, and **broad agreement** still need to be agreed on before the dashboard metrics are finalized.
- The public data may not contain all post-level characteristics or internal exposure/impression information needed to explain why some requests or notes receive less activity.

## Handoff to Task 2 — Source & Scope

Task 2 can use this notebook together with `technical.md` to summarize:

1. the public data source,
2. the four tables inspected,
3. what one row represents in each table,
4. the 1,000-row Phase 1 exploration scope,
5. the key IDs and how the tables connect.

No separate smaller data files need to be committed to the repository.
